In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# filenames
files = {
    "metric.csv": "kpi_name",
    "trace.csv": "trace_name",
    "log.csv": "log_name",
    "error_logs.csv": "message",  # error_logs has no KPI field; use 'message' for summary
}

# read files into dataframes (reuse variable df for efficiency)
dfs = {}
for fname in files:
    path = Path(fname)
    if path.exists():
        dfs[fname] = pd.read_csv(path)
    else:
        dfs[fname] = pd.DataFrame()  # empty placeholder if missing

# helper to convert timestamp bounds to Asia/Shanghai timezone strings
def ts_bounds(series):
    if series is None or series.empty:
        return (pd.NA, pd.NA)
    # convert epoch seconds to UTC then to Asia/Shanghai
    try:
        dt = pd.to_datetime(series.astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
        return (dt.min().isoformat(), dt.max().isoformat())
    except Exception:
        return (pd.NA, pd.NA)

# build summary rows
rows = []
for fname, name_col in files.items():
    df = dfs[fname]
    row = {"filename": fname}
    if df.empty:
        row.update({
            "rows": 0,
            "unique_cmdb_count": 0,
            "unique_cmdb_list": [],
            "unique_name_count": 0,
            "unique_name_list": [],
            "min_timestamp": pd.NA,
            "max_timestamp": pd.NA,
        })
    else:
        rows_count = int(df.shape[0])
        # cmdb_id unique list (limit 30)
        if "cmdb_id" in df.columns:
            uniq_cmdb = pd.Series(df["cmdb_id"].astype(str).unique())
            uniq_cmdb_list = uniq_cmdb.iloc[:30].tolist()
            uniq_cmdb_count = int(uniq_cmdb.size)
        else:
            uniq_cmdb_list = []
            uniq_cmdb_count = 0

        # name column unique list (limit 50)
        if name_col in df.columns:
            # truncate long strings for safety
            def _truncate(x, n=120):
                s = str(x)
                return s if len(s) <= n else s[:n] + "…"
            uniq_names = pd.Series(df[name_col].fillna("").astype(str).unique()).apply(_truncate)
            uniq_name_list = uniq_names.iloc[:50].tolist()
            uniq_name_count = int(uniq_names.size)
        else:
            uniq_name_list = []
            uniq_name_count = 0

        # timestamp bounds
        if "timestamp" in df.columns and not df["timestamp"].dropna().empty:
            min_ts, max_ts = ts_bounds(df["timestamp"])
        else:
            min_ts, max_ts = (pd.NA, pd.NA)

        row.update({
            "rows": rows_count,
            "unique_cmdb_count": uniq_cmdb_count,
            "unique_cmdb_list": uniq_cmdb_list,
            "unique_name_count": uniq_name_count,
            "unique_name_list": uniq_name_list,
            "min_timestamp": min_ts,
            "max_timestamp": max_ts,
        })
    rows.append(row)

summary_df = pd.DataFrame(rows, columns=[
    "filename", "rows", "unique_cmdb_count", "unique_cmdb_list",
    "unique_name_count", "unique_name_list", "min_timestamp", "max_timestamp"
])

# keep the final displayed output compact
summary_df

```
Out[1]:
```
```python
# Summarize the previously computed summary_df into a plain-English compact summary string.
# Reuse summary_df if available; otherwise, fallback to reading files (robustness).
import pandas as pd
from pathlib import Path

if 'summary_df' not in globals():
    # fallback: recreate the compact summary table (minimal work)
    files = {
        "metric.csv": "kpi_name",
        "trace.csv": "trace_name",
        "log.csv": "log_name",
        "error_logs.csv": "message",
    }
    rows = []
    for fname, name_col in files.items():
        path = Path(fname)
        if not path.exists():
            rows.append({
                "filename": fname, "rows": 0, "unique_cmdb_count": 0, "unique_cmdb_list": [],
                "unique_name_count": 0, "unique_name_list": [], "min_timestamp": pd.NA, "max_timestamp": pd.NA
            })
            continue
        df = pd.read_csv(path)
        if df.empty:
            rows.append({
                "filename": fname, "rows": 0, "unique_cmdb_count": 0, "unique_cmdb_list": [],
                "unique_name_count": 0, "unique_name_list": [], "min_timestamp": pd.NA, "max_timestamp": pd.NA
            })
            continue
        uniq_cmdb = pd.Series(df["cmdb_id"].astype(str).unique()) if "cmdb_id" in df.columns else pd.Series([])
        uniq_names = pd.Series(df[name_col].fillna("").astype(str).unique()) if name_col in df.columns else pd.Series([])
        min_ts = pd.to_datetime(df["timestamp"].astype(int), unit='s', utc=True).dt.tz_convert("Asia/Shanghai").min().isoformat() if "timestamp" in df.columns and not df["timestamp"].dropna().empty else pd.NA
        max_ts = pd.to_datetime(df["timestamp"].astype(int), unit='s', utc=True).dt.tz_convert("Asia/Shanghai").max().isoformat() if "timestamp" in df.columns and not df["timestamp"].dropna().empty else pd.NA
        rows.append({
            "filename": fname,
            "rows": int(df.shape[0]),
            "unique_cmdb_count": int(uniq_cmdb.size),
            "unique_cmdb_list": uniq_cmdb.iloc[:30].tolist(),
            "unique_name_count": int(uniq_names.size),
            "unique_name_list": uniq_names.iloc[:50].tolist(),
            "min_timestamp": min_ts,
            "max_timestamp": max_ts,
        })
    summary_df = pd.DataFrame(rows)

# Build concise plain-English summary
lines = []
for _, r in summary_df.iterrows():
    fname = r['filename']
    rows_count = int(r['rows'])
    cmdb_count = int(r['unique_cmdb_count'])
    cmdb_examples = ", ".join(r['unique_cmdb_list'][:6]) + (", ..." if cmdb_count > 6 else "") if r['unique_cmdb_list'] else "none"
    name_count = int(r['unique_name_count'])
    name_examples = ", ".join(r['unique_name_list'][:6]) + (", ..." if name_count > 6 else "") if r['unique_name_list'] else "none"
    min_ts = r['min_timestamp'] if pd.notna(r['min_timestamp']) else "N/A"
    max_ts = r['max_timestamp'] if pd.notna(r['max_timestamp']) else "N/A"
    line = (f"{fname}: {rows_count} rows; {cmdb_count} unique cmdb_id (examples: {cmdb_examples}); "
            f"{name_count} unique names (examples: {name_examples}); time range {min_ts} to {max_ts}.")
    lines.append(line)

summary_text = "Summary of telemetry files:\n" + "\n".join(lines)

# Display the final concise summary string
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

filename   rows  unique_cmdb_count                                   unique_cmdb_list  unique_name_count                                   unique_name_list              min_timestamp              max_timestamp
0      metric.csv  26930                 27  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                349  [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM...  2021-03-10T03:30:00+08:00  2021-03-10T04:00:00+08:00
1       trace.csv   2910                 12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...                 69  [trace.self.duration_mean, trace.self.duration...  2021-03-10T03:30:00+08:00  2021-03-10T03:59:00+08:00
2         log.csv    257                  7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...                  3  [log.error_count, log.row_count, log.gc_pause_...  2021-03-10T03:30:00+08:00  2021-03-10T04:00:00+08:00
3  error_logs.csv      0                  0                                                 []                  0                                                 []                       <NA>                       <NA>```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# load metric.csv (reuse metric_df if already in memory)
if 'metric_df' not in globals():
    metric_path = Path("metric.csv")
    metric_df = pd.read_csv(metric_path)
else:
    metric_df = metric_df  # reuse

# ensure correct dtypes
metric_df = metric_df.copy()
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Global summary per (cmdb_id, kpi_name) across full series (do NOT filter by time)
grouped = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
group_stats = grouped.agg(
    total_points = 'count',
    min = 'min',
    max = 'max',
    mean = 'mean',
    p95 = lambda x: x.quantile(0.95),
    p99 = lambda x: x.quantile(0.99)
).reset_index()

# 2) Define incident window in Asia/Shanghai and convert to epoch seconds (UTC-based)
start_local = pd.Timestamp("2021-03-10 03:30:00", tz=tz)
end_local   = pd.Timestamp("2021-03-10 04:00:00", tz=tz)
# convert to UTC epoch seconds
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter metric_df to the incident window (inclusive)
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 4) Merge global p95 into window_df to detect anomalies (value >= global p95)
window_with_p95 = window_df.merge(group_stats[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['p95']

# 5) Aggregate within window per group: count_in_window (where value >= p95), earliest anomaly ts, and max value in window
agg_window = window_with_p95.groupby(['cmdb_id','kpi_name']).agg(
    count_in_window = ('is_anom', 'sum'),
    earliest_anom_ts = ('timestamp', lambda s: s[window_with_p95.loc[s.index, 'is_anom']].min() if s.size>0 else pd.NA),
    max_value_in_window = ('value', 'max'),
).reset_index()

# Ensure earliest_anom_ts NA for groups without anomalies
agg_window['earliest_anom_ts'] = agg_window['earliest_anom_ts'].where(agg_window['count_in_window']>0, pd.NA)

# 6) Merge global stats with window aggregates
result = group_stats.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no window data
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].where(result['max_value_in_window'].notna(), pd.NA)

# Convert earliest_anom_ts to Asia/Shanghai ISO strings
def ts_to_iso(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).isoformat()

result['earliest_anomaly_timestamp_in_window'] = result['earliest_anom_ts'].apply(ts_to_iso)

# 7) Select and rename final columns as requested
final_cols = [
    'cmdb_id',
    'kpi_name',
    'total_points',
    'p95',
    'p99',
    'min',
    'max',
    'mean',
    'count_in_window',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]
final_df = result[final_cols].copy()

# 8) Sort by count_in_window desc, then earliest_anomaly_timestamp_in_window asc (NA last)
# For sorting, convert earliest timestamps back to datetimes; NA will be placed at the end
earliest_dt = pd.to_datetime(final_df['earliest_anomaly_timestamp_in_window'].dropna()).dt.tz_convert(tz)
# create a sort key series aligning with final_df index
def make_sort_key(col):
    # convert iso strings to datetime64[ns] for sorting; NA becomes large value
    return pd.to_datetime(final_df['earliest_anomaly_timestamp_in_window']).fillna(pd.Timestamp.max.tz_localize(tz))
sort_key = make_sort_key('earliest_anomaly_timestamp_in_window')

final_df['_sort_earliest'] = sort_key
final_df_sorted = final_df.sort_values(by=['count_in_window', '_sort_earliest'], ascending=[False, True]).drop(columns=['_sort_earliest'])

# 9) Limit to top 20 rows and keep output compact
top20 = final_df_sorted.head(20)

# Display the compact table (at most 20 rows)
top20

```
Out[2]:
```
```python
# Build a concise plain-English summary of the previous metric analysis and display it.
import pandas as pd

# Reuse top20 if available; otherwise load minimal from disk (robustness)
if 'top20' not in globals():
    df = pd.read_csv("metric.csv")
    # minimal fallback: no recomputation, so show a note
    summary_text = "Top20 result not found in the environment; please re-run the analysis step to produce top20."
else:
    df = top20.copy()
    n_rows = df.shape[0]
    # summary stats from the displayed top20
    count_in_window_vals = df['count_in_window'].unique().tolist()
    earliest_vals = df['earliest_anomaly_timestamp_in_window'].dropna().unique().tolist()
    # examples of the top (cmdb_id, kpi_name)
    examples = df[['cmdb_id','kpi_name']].head(10).apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']}", axis=1).tolist()
    # common patterns
    p95_equals_max = (df['p95'] == df['max']).all()
    constant_count_all = (df['count_in_window'] == df['total_points']).all()
    earliest_example = earliest_vals[0] if earliest_vals else "N/A"

    summary_lines = [
        "Metric analysis summary (metric.csv):",
        f"- Computed global stats (count, min, max, mean, P95, P99) per (cmdb_id, kpi_name) across the full series.",
        "- Incident window: 2021-03-10 03:30:00 to 2021-03-10 04:00:00 (Asia/Shanghai).",
        f"- Top {n_rows} (cmdb_id, kpi_name) entries (sorted by count_in_window):",
        f"  • Examples: {', '.join(examples)}",
        f"- Observations:",
        f"  • count_in_window values present in top entries: {count_in_window_vals}.",
        f"  • Earliest anomaly timestamps in-window (examples): {earliest_example} (most top entries share this start time).",
        f"  • For these top entries, {'P95 equals max across the full series' if p95_equals_max else 'P95 does not always equal max'};",
        f"    {'and every point in the 31-minute window met/exceeded P95' if constant_count_all else 'and not all window points met/exceeded P95'}.",
        "- Notable KPI types in top entries: JVM memory metrics (HeapMemoryMax, NoHeapMemoryUsed) and OS Linux metrics (network TCP CLOSE/FIN WAIT, process counts).",
        "- Interpretation: The top series were at or above their historical P95 throughout the incident window (many values constant at their high/end values).",
        "- Suggested next steps: inspect application traces/logs and longer historical context for these specific KPIs to determine if values represent an actual anomaly or expected/constant config values."
    ]
    summary_text = "\n".join(summary_lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points           p95           p99           min           max          mean  count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09               31            2021-03-10T03:30:00+08:00         2.040136e+09
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31  5.789522e+07  5.789522e+07  5.789522e+07  5.789522e+07  5.789522e+07               31            2021-03-10T03:30:00+08:00         5.789522e+07
67      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
68      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
90      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09               31            2021-03-10T03:30:00+08:00         2.040136e+09
157     IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
158     IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
168     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00               31            2021-03-10T03:30:00+08:00         1.000000e+00
171     IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01               31            2021-03-10T03:30:00+08:00         3.000000e-01
172     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00               31            2021-03-10T03:30:00+08:00         3.000000e+00
180     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09               31            2021-03-10T03:30:00+08:00         2.040136e+09
183     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31  5.069049e+08  5.069049e+08  5.069049e+08  5.069049e+08  5.069049e+08               31            2021-03-10T03:30:00+08:00         5.069049e+08
272     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09               31            2021-03-10T03:30:00+08:00         2.040136e+09
275     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31  9.471166e+08  9.471166e+08  9.471166e+08  9.471166e+08  9.471166e+08               31            2021-03-10T03:30:00+08:00         9.471166e+08
575  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
576  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
578  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
582  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00               31            2021-03-10T03:30:00+08:00         7.000000e+00
802  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00
804  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31            2021-03-10T03:30:00+08:00         0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv (reuse trace_df if present)
if 'trace_df' not in globals():
    trace_path = Path("trace.csv")
    trace_df = pd.read_csv(trace_path)
else:
    trace_df = trace_df

# Ensure correct dtypes and copy for safety
trace_df = trace_df.copy()
trace_df['timestamp'] = trace_df['timestamp'].astype('int64')
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Global stats per (cmdb_id, trace_name) across the full series (compute thresholds BEFORE filtering)
grouped = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
group_stats = grouped.agg(
    total_points = 'count',
    min = 'min',
    max = 'max',
    mean = 'mean',
    p95 = lambda x: x.quantile(0.95),
    p99 = lambda x: x.quantile(0.99)
).reset_index()

# 2) Define incident window in Asia/Shanghai and convert to UTC epoch seconds
start_local = pd.Timestamp("2021-03-10 03:30:00", tz=tz)
end_local   = pd.Timestamp("2021-03-10 04:00:00", tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter trace_df to the incident window (inclusive)
window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 4) Merge global p95 into window_df (so threshold is from full series)
window_with_p95 = window_df.merge(group_stats[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['p95']

# 5) Aggregate within the window:
# - count_in_window: number of points where value >= p95
# - earliest_anomaly_timestamp_in_window: earliest timestamp where is_anom True (kept as epoch seconds for sorting)
# - max_value_in_window: max value in the window (regardless of threshold)
anom_df = window_with_p95[window_with_p95['is_anom'] == True]
anom_stats = anom_df.groupby(['cmdb_id','trace_name']).agg(
    count_in_window = ('is_anom', 'sum'),
    earliest_anom_ts = ('timestamp', 'min')  # epoch seconds UTC
).reset_index()

window_stats = window_df.groupby(['cmdb_id','trace_name']).agg(
    max_value_in_window = ('value', 'max')
).reset_index()

# 6) Merge global stats with window aggregates
result = group_stats.merge(anom_stats, on=['cmdb_id','trace_name'], how='left')
result = result.merge(window_stats, on=['cmdb_id','trace_name'], how='left')

# Fill missing values for groups without window data/anomalies
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['earliest_anom_ts'] = result['earliest_anom_ts'].where(result['count_in_window']>0, pd.NA)
result['max_value_in_window'] = result['max_value_in_window'].where(result['max_value_in_window'].notna(), pd.NA)

# Convert earliest_anom_ts (epoch seconds) to Asia/Shanghai ISO strings
def ts_to_iso_local(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()

result['earliest_anomaly_timestamp_in_window'] = result['earliest_anom_ts'].apply(ts_to_iso_local)

# 7) Select and order final columns as requested
final_cols = [
    'cmdb_id',
    'trace_name',
    'total_points',
    'p95',
    'p99',
    'min',
    'max',
    'mean',
    'count_in_window',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]
final_df = result[final_cols].copy()

# 8) Sort by count_in_window desc, then earliest_anomaly_timestamp_in_window asc (NA last)
# Create a sortable key for earliest: convert iso to datetime, NA -> large timestamp
sort_key = pd.to_datetime(final_df['earliest_anomaly_timestamp_in_window']).fillna(pd.Timestamp.max.tz_localize(tz))
final_df['_sort_earliest'] = sort_key
final_df_sorted = final_df.sort_values(by=['count_in_window', '_sort_earliest'], ascending=[False, True]).drop(columns=['_sort_earliest'])

# 9) Limit to top 20 rows and display compactly
top20_trace = final_df_sorted.head(20)

# Display the result DataFrame (compact)
top20_trace

```
Out[3]:
```
```python
# Build a concise plain-English summary of the trace analysis using the previously computed top20_trace.
# Reuse top20_trace from the environment.
import pandas as pd

if 'top20_trace' not in globals():
    summary_text = "Top20 trace results not available in the environment. Please run the trace analysis to produce top20_trace."
else:
    df = top20_trace.copy()
    total_rows = df.shape[0]
    groups_with_anom = int((df['count_in_window'] > 0).sum())
    max_count = int(df['count_in_window'].max())
    max_count_groups = int((df['count_in_window'] == max_count).sum())
    earliest_anoms = df['earliest_anomaly_timestamp_in_window'].dropna().sort_values()
    earliest_overall = earliest_anoms.iloc[0] if not earliest_anoms.empty else "N/A"
    # examples of top entries
    examples = df[['cmdb_id','trace_name','count_in_window']].head(6).apply(
        lambda r: f"{r['cmdb_id']}:{r['trace_name']} (count_in_window={r['count_in_window']})", axis=1
    ).tolist()
    # common trace name patterns in top rows
    common_patterns = df['trace_name'].value_counts().head(6).to_dict()

    summary_lines = [
        "Trace analysis summary (trace.csv):",
        "- Analysis: global stats computed per (cmdb_id, trace_name); incident window = 2021-03-10 03:30:00 to 04:00:00 (Asia/Shanghai).",
        f"- Top {total_rows} groups shown (by count_in_window). {groups_with_anom} groups had values >= global P95 within the window.",
        f"- Maximum count_in_window = {max_count}, observed in {max_count_groups} group(s).",
        f"- Earliest anomaly timestamp among these top groups: {earliest_overall}.",
        f"- Example top entries: {', '.join(examples)}.",
        f"- Common trace types in top entries: {', '.join([f'{k}({v})' for k,v in common_patterns.items()])}.",
        "- Observation: anomalies are concentrated near 03:39–03:42 (Asia/Shanghai) and involve row_count and duration metrics (to/from and self traces).",
        "- Interpretation: several services show elevated trace durations or row counts during the incident window; these are candidates for deeper investigation (service endpoints and correlated logs)."
    ]
    summary_text = "\n".join(summary_lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  total_points        p95        p99       min         max       mean  count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
86       MG02    trace.from_dockerA2.row_count             6   3.000000   3.000000  1.000000    3.000000   2.333333                4            2021-03-10T03:42:00+08:00             3.000000
197  dockerA2          trace.to_MG02.row_count             6   3.000000   3.000000  1.000000    3.000000   2.333333                4            2021-03-10T03:42:00+08:00             3.000000
0        IG01         trace.self.duration_mean            23   0.079780   0.649140  0.016000    0.809667   0.073703                2            2021-03-10T03:39:00+08:00             0.809667
1        IG01          trace.self.duration_p95            23   0.240750   1.749535  0.016000    2.173750   0.163011                2            2021-03-10T03:39:00+08:00             2.173750
15       IG02         trace.self.duration_mean            24   0.068819   0.164066  0.016000    0.192500   0.041740                2            2021-03-10T03:39:00+08:00             0.192500
16       IG02          trace.self.duration_p95            24   0.190005   0.298260  0.016000    0.330000   0.067300                2            2021-03-10T03:39:00+08:00             0.330000
93       MG02         trace.self.duration_mean            23   0.065450   0.445282  0.007000    0.552400   0.047536                2            2021-03-10T03:39:00+08:00             0.552400
94       MG02          trace.self.duration_p95            23   0.140820   1.430184  0.008000    1.793800   0.124472                2            2021-03-10T03:39:00+08:00             1.793800
2        IG01             trace.self.row_count            23  29.600000  70.560000  2.000000   82.000000  12.434783                2            2021-03-10T03:40:00+08:00            82.000000
3        IG01  trace.to_Tomcat01.duration_mean            16   0.076000   0.076000  0.012000    0.076000   0.038107                2            2021-03-10T03:40:00+08:00             0.076000
17       IG02             trace.self.row_count            24  31.400000  65.880000  2.000000   76.000000  11.916667                2            2021-03-10T03:40:00+08:00            76.000000
23       IG02      trace.to_Tomcat02.row_count            16   6.000000   6.000000  1.000000    6.000000   2.000000                2            2021-03-10T03:40:00+08:00             6.000000
54       MG01         trace.self.duration_mean            25   0.034984   0.045890  0.005500    0.049118   0.018367                2            2021-03-10T03:40:00+08:00             0.049118
98       MG02      trace.to_dockerA1.row_count             9   6.000000   6.000000  1.000000    6.000000   2.333333                2            2021-03-10T03:40:00+08:00             6.000000
101      MG02      trace.to_dockerA2.row_count            10   8.000000   8.000000  1.000000    8.000000   3.700000                2            2021-03-10T03:40:00+08:00             8.000000
108  Tomcat01    trace.from_IG01.duration_mean            16   0.076000   0.076000  0.012000    0.076000   0.038107                2            2021-03-10T03:40:00+08:00             0.076000
128  Tomcat02        trace.from_IG02.row_count            16   6.000000   6.000000  1.000000    6.000000   2.000000                2            2021-03-10T03:40:00+08:00             6.000000
129  Tomcat02         trace.self.duration_mean            21   0.009062   0.009679  0.001833    0.009833   0.004653                2            2021-03-10T03:40:00+08:00             0.009833
130  Tomcat02          trace.self.duration_p95            21   0.064900   0.065980  0.007750    0.066250   0.023095                2            2021-03-10T03:40:00+08:00             0.066250
131  Tomcat02             trace.self.row_count            21  60.000000  95.200000  6.000000  104.000000  21.238095                2            2021-03-10T03:40:00+08:00           104.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Load log.csv (reuse log_df if present)
if 'log_df' not in globals():
    log_path = Path("log.csv")
    log_df = pd.read_csv(log_path)
else:
    log_df = log_df

# Prepare dataframe
log_df = log_df.copy()
log_df['timestamp'] = log_df['timestamp'].astype('int64')
log_df['cmdb_id'] = log_df['cmdb_id'].astype(str)
log_df['log_name'] = log_df['log_name'].astype(str)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Global stats per (cmdb_id, log_name) across full series (thresholds computed before filtering)
grouped = log_df.groupby(['cmdb_id', 'log_name'])['value']
global_stats = grouped.agg(
    total_points = 'count',
    min = 'min',
    max = 'max',
    mean = 'mean',
    p95 = lambda x: x.quantile(0.95),
    p5  = lambda x: x.quantile(0.05)
).reset_index()

# 2) Define incident window in Asia/Shanghai and convert to UTC epoch seconds
start_local = pd.Timestamp("2021-03-10 03:30:00", tz=tz)
end_local   = pd.Timestamp("2021-03-10 04:00:00", tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter log_df to the incident window (inclusive)
window_df = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 4) Merge global p95 and p5 into window_df (thresholds from full series)
window_with_thr = window_df.merge(global_stats[['cmdb_id','log_name','p95','p5']], on=['cmdb_id','log_name'], how='left')
window_with_thr['is_high'] = window_with_thr['value'] >= window_with_thr['p95']
window_with_thr['is_low']  = window_with_thr['value'] <= window_with_thr['p5']

# 5) Aggregate high/low anomaly info within the window
high_df = window_with_thr[window_with_thr['is_high'] == True]
high_stats = high_df.groupby(['cmdb_id','log_name']).agg(
    count_high_in_window = ('is_high', 'sum'),
    earliest_high_ts = ('timestamp', 'min')
).reset_index()

low_df = window_with_thr[window_with_thr['is_low'] == True]
low_stats = low_df.groupby(['cmdb_id','log_name']).agg(
    count_low_in_window = ('is_low', 'sum'),
    earliest_low_ts = ('timestamp', 'min')
).reset_index()

# Window-level max/min values (regardless of thresholds)
window_vals = window_df.groupby(['cmdb_id','log_name']).agg(
    max_value_in_window = ('value', 'max'),
    min_value_in_window = ('value', 'min')
).reset_index()

# 6) Merge everything with global_stats
result = global_stats.merge(high_stats, on=['cmdb_id','log_name'], how='left')
result = result.merge(low_stats, on=['cmdb_id','log_name'], how='left')
result = result.merge(window_vals, on=['cmdb_id','log_name'], how='left')

# 7) Fill missing counts with 0 and ensure timestamp fields are NA where appropriate
result['count_high_in_window'] = result['count_high_in_window'].fillna(0).astype(int)
result['count_low_in_window'] = result['count_low_in_window'].fillna(0).astype(int)
result['earliest_high_ts'] = result['earliest_high_ts'].where(result['count_high_in_window']>0, pd.NA)
result['earliest_low_ts']  = result['earliest_low_ts'].where(result['count_low_in_window']>0, pd.NA)
result['max_value_in_window'] = result['max_value_in_window'].where(result['max_value_in_window'].notna(), pd.NA)
result['min_value_in_window'] = result['min_value_in_window'].where(result['min_value_in_window'].notna(), pd.NA)

# 8) Convert earliest timestamps to Asia/Shanghai ISO strings, keep NA as null
def ts_to_iso_local(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()

result['earliest_high_timestamp_in_window'] = result['earliest_high_ts'].apply(ts_to_iso_local)
result['earliest_low_timestamp_in_window']  = result['earliest_low_ts'].apply(ts_to_iso_local)

# 9) Select final columns in requested order
final_cols = [
    'cmdb_id',
    'log_name',
    'total_points',
    'p95',
    'p5',
    'min',
    'max',
    'mean',
    'count_high_in_window',
    'earliest_high_timestamp_in_window',
    'count_low_in_window',
    'earliest_low_timestamp_in_window',
    'max_value_in_window',
    'min_value_in_window'
]
final_df = result[final_cols].copy()

# 10) Sorting: by count_high_in_window desc, then count_low_in_window desc, then earliest_high_timestamp_in_window asc (NA last)
# Create sortable key for earliest_high (NA -> very large timestamp)
sort_key_earliest_high = pd.to_datetime(final_df['earliest_high_timestamp_in_window']).fillna(pd.Timestamp.max.tz_localize(tz))
final_df['_sort_earliest_high'] = sort_key_earliest_high

final_df_sorted = final_df.sort_values(
    by=['count_high_in_window', 'count_low_in_window', '_sort_earliest_high'],
    ascending=[False, False, True]
).drop(columns=['_sort_earliest_high'])

# 11) Limit to top 20 rows (compact) and display
top20_logs = final_df_sorted.head(20)

top20_logs

```
Out[4]:
```
```python
# Compose a concise plain-English summary of the log analysis using the previously computed top20_logs.
# The final output is the summary string (displayed as the cell result).
if 'top20_logs' in globals():
    df = top20_logs.copy()
    examples = df[['cmdb_id','log_name']].head(8).apply(lambda r: f"{r['cmdb_id']}:{r['log_name']}", axis=1).tolist()
    # Count how many strict "all-zero" error_count patterns exist in top results
    is_error_count = df['log_name'] == 'log.error_count'
    zero_error_rows = df[is_error_count & (df['min'] == 0) & (df['max'] == 0)]
    zero_error_examples = zero_error_rows[['cmdb_id','log_name']].apply(lambda r: f"{r['cmdb_id']}", axis=1).tolist()
    # Row_count highlights
    rowcount_rows = df[df['log_name'] == 'log.row_count'][['cmdb_id','count_high_in_window','earliest_high_timestamp_in_window','count_low_in_window','earliest_low_timestamp_in_window','max_value_in_window','min_value_in_window']].head(6)
    rowcount_examples = rowcount_rows.apply(lambda r: f"{r['cmdb_id']} (highs={r['count_high_in_window']} earliest_high={r['earliest_high_timestamp_in_window']}, lows={r['count_low_in_window']} earliest_low={r['earliest_low_timestamp_in_window']}, max={r['max_value_in_window']})", axis=1).tolist()
    # GC pause highlights
    gc_rows = df[df['log_name'].str.contains('gc_pause')][['cmdb_id','count_high_in_window','earliest_high_timestamp_in_window']].head(6)
    gc_examples = gc_rows.apply(lambda r: f"{r['cmdb_id']} (highs={r['count_high_in_window']} at {r['earliest_high_timestamp_in_window']})", axis=1).tolist()

    summary_lines = [
        "Log analysis summary (log.csv):",
        "- Global stats computed per (cmdb_id, log_name). Incident window: 2021-03-10 03:30:00 to 04:00:00 (Asia/Shanghai).",
        f"- Top groups examples: {', '.join(examples)}.",
        "",
        "Key observations:",
        f"- Several services show log.error_count constantly zero during the window (examples: {', '.join(zero_error_examples)}). For these, p95=p5=0 and every point in-window equals that value (count_high_in_window == total_points and count_low_in_window == total_points).",
        f"- log.row_count shows notable variability and both high and low occurrences in the window. Examples: {', '.join(rowcount_examples)}.",
        f"- GC pause metrics appear in the top list with occasional high values during the window (examples: {', '.join(gc_examples)}).",
        "",
        "Interpretation and next steps:",
        "- Constant zero error_count likely indicates no errors were recorded (may be normal baseline); these are not necessarily anomalies.",
        "- The row_count entries show spikes (highs typically around 03:40) and troughs (lows around 03:30–03:44) for several Tomcat/Apache instances — these are candidate signals to investigate (request rates, upstream calls, or batching).",
        "- GC pause occurrences should be inspected if correlated with throughput drops or increased latencies.",
        "- Suggested follow-ups: correlate these log events with trace and metric anomalies, and inspect application logs around the highlighted timestamps (especially ~03:30 and ~03:40)."
    ]
    summary = "\n".join(summary_lines)
else:
    summary = "top20_logs not found in environment; please re-run the log analysis to produce the top20_logs dataframe."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  total_points      p95      p5   min    max      mean  count_high_in_window earliest_high_timestamp_in_window  count_low_in_window earliest_low_timestamp_in_window  max_value_in_window  min_value_in_window
2   Tomcat01   log.error_count            22   0.0000  0.0000  0.00   0.00  0.000000                    22         2021-03-10T03:30:00+08:00                   22        2021-03-10T03:30:00+08:00                 0.00                 0.00
5   Tomcat02   log.error_count            21   0.0000  0.0000  0.00   0.00  0.000000                    21         2021-03-10T03:30:00+08:00                   21        2021-03-10T03:30:00+08:00                 0.00                 0.00
14  apache01   log.error_count            21   0.0000  0.0000  0.00   0.00  0.000000                    21         2021-03-10T03:30:00+08:00                   21        2021-03-10T03:30:00+08:00                 0.00                 0.00
11  Tomcat04   log.error_count            20   0.0000  0.0000  0.00   0.00  0.000000                    20         2021-03-10T03:30:00+08:00                   20        2021-03-10T03:30:00+08:00                 0.00                 0.00
16  apache02   log.error_count            20   0.0000  0.0000  0.00   0.00  0.000000                    20         2021-03-10T03:30:00+08:00                   20        2021-03-10T03:30:00+08:00                 0.00                 0.00
8   Tomcat03   log.error_count            19   0.0000  0.0000  0.00   0.00  0.000000                    19         2021-03-10T03:30:00+08:00                   19        2021-03-10T03:30:00+08:00                 0.00                 0.00
4   Tomcat01     log.row_count            22   6.0000  1.0000  1.00  22.00  3.227273                     3         2021-03-10T03:40:00+08:00                    9        2021-03-10T03:32:00+08:00                22.00                 1.00
7   Tomcat02     log.row_count            21  10.0000  1.0000  1.00  16.00  3.380952                     2         2021-03-10T03:40:00+08:00                    7        2021-03-10T03:30:00+08:00                16.00                 1.00
15  apache01     log.row_count            21  15.0000  1.0000  1.00  40.00  6.714286                     2         2021-03-10T03:40:00+08:00                    3        2021-03-10T03:44:00+08:00                40.00                 1.00
0    Mysql01   log.error_count             2   0.0000  0.0000  0.00   0.00  0.000000                     2         2021-03-10T03:30:00+08:00                    2        2021-03-10T03:30:00+08:00                 0.00                 0.00
3   Tomcat01  log.gc_pause_max             2   0.0700  0.0700  0.07   0.07  0.070000                     2         2021-03-10T03:37:00+08:00                    2        2021-03-10T03:37:00+08:00                 0.07                 0.07
13  Tomcat04     log.row_count            20   8.2500  1.0000  1.00  13.00  3.350000                     1         2021-03-10T03:42:00+08:00                    7        2021-03-10T03:30:00+08:00                13.00                 1.00
10  Tomcat03     log.row_count            19  12.7000  1.0000  1.00  28.00  4.421053                     1         2021-03-10T03:42:00+08:00                    6        2021-03-10T03:30:00+08:00                28.00                 1.00
17  apache02     log.row_count            20  17.1500  2.0000  2.00  39.00  7.250000                     1         2021-03-10T03:42:00+08:00                    6        2021-03-10T03:30:00+08:00                39.00                 2.00
12  Tomcat04  log.gc_pause_max             2   0.0895  0.0805  0.08   0.09  0.085000                     1         2021-03-10T03:30:00+08:00                    1        2021-03-10T03:53:00+08:00                 0.09                 0.08
9   Tomcat03  log.gc_pause_max             1   0.0700  0.0700  0.07   0.07  0.070000                     1         2021-03-10T03:46:00+08:00                    1        2021-03-10T03:46:00+08:00                 0.07                 0.07
6   Tomcat02  log.gc_pause_max             2   0.0890  0.0710  0.07   0.09  0.080000                     1         2021-03-10T03:59:00+08:00                    1        2021-03-10T03:33:00+08:00                 0.09                 0.07
1    Mysql01     log.row_count             2   7.6500  1.3500  1.00   8.00  4.500000                     1         2021-03-10T04:00:00+08:00                    1        2021-03-10T03:30:00+08:00                 8.00                 1.00```
```